In [3]:
import sys
print(sys.version)

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [1]:
!pip install mediapipe==0.10.14 opencv-python pandas scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import cv2
import os
import numpy as np
import mediapipe as mp
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [2]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [3]:

cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        # Make Detections
        results = holistic.process(image)
        # print(results.face_landmarks)
        
        # face_landmarks, pose_landmarks, left_hand_landmarks, right_hand_landmarks
        
        # Recolor image back to BGR for rendering
        image.flags.writeable =True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # 1. Draw face landmarks
        # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, 
        #                          mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
        #                          mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        #                          )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                                 )

        # 3. Left Hand
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                                 )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                 )
                        
        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

C:\Users\MJ\Desktop\humanaction\my_venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [4]:
num_coords= len(results.pose_landmarks.landmark)

In [5]:
landmarks= ["class"]
for val in range(1, num_coords+1):
    landmarks += ['x{}'.format(val), 'y{}'.format(val), 'z{}'.format(val), 'v{}'.format(val)]

In [9]:
with open('coords_.csv', mode='w', newline='') as f:
    csv_writer = csv.writer(f, delimiter=',', quoting=csv.QUOTE_MINIMAL)
    csv_writer.writerow(landmarks)

In [18]:
class_name = 'Jumping'

In [19]:
cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        # Make Detections
        results = holistic.process(image)
        # print(results.face_landmarks)
        
        # face_landmarks, pose_landmarks, left_hand_landmarks, right_hand_landmarks
        
        # Recolor image back to BGR for rendering
        image.flags.writeable =True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # # 1. Draw face landmarks
        # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, 
        #                          mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
        #                          mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        #                          )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                                 )

        # 3. Left Hand
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                                 )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                 )

        #export coordinates
        try:
            #Extract pose landmarks
            pose = results.pose_landmarks.landmark
            pose_row = list(np.array([[landmark.x, landmark.y, landmark.z, landmark.visibility] for landmark in pose]).flatten())
            
            pose_row.insert(0, class_name)
            #export to csv
            with open('coords_.csv', mode='a', newline='') as f:
                csv_writer = csv.writer(f, delimiter=',', quoting=csv.QUOTE_MINIMAL)
                csv_writer.writerow(pose_row)
                
        except:
            pass
                        
        cv2.imshow('Raw Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [44]:
#data processing and model training

df1 = pd.read_csv('coords.csv')
df2 = pd.read_csv('coords_.csv')
df1.shape, df2.shape

((1834, 133), (2014, 133))

In [45]:
#concatenate the 2 dataframes
df2.replace('Jumped', 'Jumping', inplace=True)

,class,x1,y1,z1,v1,x2,y2,z2,v2,x3,...,z31,v31,x32,y32,z32,v32,x33,y33,z33,v33
0,Clapping,0.652315,0.119390,-2.922270,0.995507,0.699870,0.056641,-2.908461,0.993155,0.723756,...,-0.161190,0.247447,0.501663,1.333622,-0.513806,0.169682,0.284553,1.355469,-0.618832,0.157323
1,Clapping,0.692673,0.113388,-2.471931,0.993463,0.731862,0.027530,-2.481075,0.989032,0.760586,...,2.413127,0.242766,0.568223,1.071085,2.565081,0.166039,0.387392,1.257477,2.128788,0.160374
2,Clapping,0.688963,0.261485,-2.406460,0.986296,0.728375,0.164547,-2.446679,0.973325,0.759831,...,1.984725,0.237183,0.610960,2.100121,1.994503,0.162929,0.369349,2.160030,1.494595,0.159758
3,Clapping,0.687281,0.221397,-1.276072,0.987032,0.726542,0.138638,-1.251219,0.974581,0.756493,...,1.294709,0.213466,0.652728,2.806927,1.188642,0.146659,0.444750,2.799432,0.630332,0.143790
4,Clapping,0.685215,0.240418,-1.483938,0.986368,0.720429,0.160359,-1.468975,0.972554,0.743482,...,1.274781,0.192413,0.618127,2.732745,0.603495,0.132771,0.376751,2.692607,0.194131,0.130051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2009,Jumping,0.565721,0.250100,-1.010707,0.970498,0.589709,0.198283,-0.985902,0.961272,0.604084,...,-0.261011,0.503121,0.568967,1.157563,-0.447346,0.500788,0.414037,1.201355,-0.499256,0.510402
2010,Jumping,0.575997,0.334780,-1.513276,0.973419,0.607510,0.275162,-1.485112,0.965097,0.625169,...,-0.137452,0.467896,0.607529,1.233291,-0.230911,0.458471,0.438003,1.288660,-0.359949,0.467658
2011,Jumping,0.575480,0.349524,-1.679886,0.976004,0.615863,0.273414,-1.687298,0.968437,0.634937,...,1.094256,0.425915,0.612671,1.267935,0.899380,0.419393,0.441089,1.246786,0.984579,0.425332
2012,Jumping,0.581678,0.362324,-1.555142,0.977492,0.618891,0.289385,-1.577875,0.968095,0.637413,...,1.315948,0.391771,0.609734,1.389005,1.282122,0.383059,0.448089,1.405723,1.094276,0.391769


In [46]:
df2['class'].value_counts()

class
Dancing         638
Clapping        506
Crossed Arms    468
Jumping         402
Name: count, dtype: int64

In [49]:
#check the class distribution
df1['class'].value_counts()

class
Dancing         465
Crossed Arms    462
Jumping         460
Clapping        447
Name: count, dtype: int64

In [50]:
df = pd.concat([df, df2], axis=0, ignore_index=True)

In [51]:
df['class'].value_counts()

class
Dancing         1103
Clapping         953
Crossed Arms     930
Jumping          862
Name: count, dtype: int64

In [52]:
#prepare data for splitting
X = df.drop('class', axis=1)
y = df['class']
y


0       Crossed Arms
1       Crossed Arms
2       Crossed Arms
3       Crossed Arms
4       Crossed Arms
            ...     
3843         Jumping
3844         Jumping
3845         Jumping
3846         Jumping
3847         Jumping
Name: class, Length: 3848, dtype: str

In [53]:
#perform splitting- train_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)

In [54]:
#check out the shape of the splits
print(f"X_train shape is ", X_train.shape)
print(f"y_train shape is ", y_train.shape)
print(f"X_test shape is ", X_test.shape)
print(f"y_test shape is ", y_test.shape)

X_train shape is  (2693, 132)
y_train shape is  (2693,)
X_test shape is  (1155, 132)
y_test shape is  (1155,)


In [57]:
pipelines={
    'lr' : make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'rc' : make_pipeline(StandardScaler(), RidgeClassifier()),
    'rf' : make_pipeline(StandardScaler(), RandomForestClassifier()),
    'gb' : make_pipeline(StandardScaler(), GradientBoostingClassifier()),
}

In [58]:
fit_models ={}
for algo, pipeline in pipelines.items():
    model = pipeline.fit(X_train, y_train)
    fit_models[algo] = model

In [ ]:
fit_models['rc'].predict(X_test)

In [ ]:
# def zoom_out(frame, scale=1.5):
#     """scale > 1 = zoom out, scale < 1 = zoom in"""
#     h, w = frame.shape[:2]
    
#     # New crop size (larger = more zoomed out)
#     new_h, new_w = int(h / scale), int(w / scale)
    
#     # Crop from center
#     top = (h - new_h) // 2
#     left = (w - new_w) // 2
#     cropped = frame[top:top+new_h, left:left+new_w]
    
#     # Resize back to original dimensions
#     return cv2.resize(cropped, (w, h))

In [59]:
from sklearn.metrics import accuracy_score
import pickle

In [60]:
for algo, model in fit_models.items():
    yhat = model.predict(X_test)
    print(algo, accuracy_score(y_test, yhat))

lr 0.9567099567099567
rc 0.929004329004329
rf 0.9896103896103896
gb 0.9930735930735931


In [61]:
with open('Human_action.pkl', 'wb') as f:
    pickle.dump(fit_models['gb'], f)